# Follow-up engineering screen definitions

Purpose: choose a common real-data microbatch for ViT and semantic-local JEPA using runtime alone, and certify that accumulation preserves the weighted loss and gradients. This notebook is an importable definition module; the Notebook 04 controller explicitly calls its exports after acquiring the GPU lease.

Inputs: the pinned Food101 train/development manifests, original image audit and existing FP32 operators. Outputs: engineering and scaling JSON evidence under the controller artifact directory, plus full-state calibration checkpoints under `checkpoints/<run_id>/engineering/`. Expected runtime: engineering under 8 minutes plus an optional 3-minute synthetic scale screen, all bounded by the parent deadline. Calibration weights are discarded; official test pixels are never read.

Resume: rerun `run_engineering_screen(ROOT, artifact_dir, identical_deadline, identical_config)` with the same artifact identity; completed cases are checked and reused and partial cases resume from full-state checkpoints. Changing configuration or exported source requires a new run ID. Scope is numerical/engineering evidence, not representation-quality evidence.

In [ ]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
from pathlib import Path
import json
ROOT = Path("/home/benjamin/workspace/ahr_go/fmm_jeppa")
for cell in json.loads((ROOT / "fmm_jepa/bootstrap.ipynb").read_text())["cells"]:
    if cell["cell_type"] == "code" and "ahr-export" in cell.get("metadata", {}).get("tags", []):
        exec(compile("".join(cell["source"]), "bootstrap#ahr-export", "exec"))
install_notebook_package(ROOT)


In [ ]:
from pathlib import Path
import copy, gc, hashlib, json, statistics, time
import numpy as np
import torch
from fmm_jepa.semantic_pilot import (
    configure_environment, read_config, reuse_food_bundle, initialize_models, run_model,
    deadline_guard, build_model,
)
from fmm_jepa.food101_training import ConsumedSampler, isolated_rng, validate_training_manifest
from fmm_jepa.food101_data import make_batch
from fmm_jepa.training import atomic_json, definition_hashes, digest_file, move_masks
from fmm_jepa.geometry import clear_geometry_cache

ENGINEERING_SCHEMA = "semantic-followup-engineering/v1"
ENGINEERING_KINDS = ("vit", "semantic")


def _engineering_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(",", ":")).encode()).hexdigest()


def _engineering_export_hash(root):
    notebook = json.loads((Path(root)/"experiments/04_followup/engineering.ipynb").read_text())
    return _engineering_hash(["".join(c["source"]) for c in notebook["cells"]
                              if c["cell_type"] == "code" and "ahr-export" in c.get("metadata", {}).get("tags", [])])


def _engineering_release(device):
    clear_geometry_cache()
    gc.collect()
    if torch.device(device).type == "cuda":
        torch.cuda.synchronize(device)
        torch.cuda.empty_cache()


def _engineering_configuration(root, config=None):
    cfg = copy.deepcopy(config) if config is not None else read_config(root)
    cfg["training"]["updates"] = 8
    cfg["training"]["checkpoint_every"] = 4
    cfg["evaluation"]["heldout_enabled"] = False
    return configure_environment(cfg)


def _trace_projection(trace):
    return {name: trace[name] for name in ("attempted_update", "sample_ids", "occurrences",
                                         "augmentation_ids", "mask_ids", "successful", "successful_update")}


def certify_microbatch_equivalence(root, artifact_dir, config, bundle, selected_microbatch, deadline_unix):
    """Actual 128-image first update: sample-average loss and all online/predictor gradients.

    Each candidate performs its own natural packed padding. Equal valid-token loss
    is averaged per sample, then block, so equal-size microbatches have equal weights.
    Padding lengths may differ. Teacher state is fixed and never receives gradients.
    """
    artifact = Path(artifact_dir)
    outpath = artifact / "microbatch_equivalence.json"
    if outpath.exists():
        result = json.loads(outpath.read_text())
        if result["selected_microbatch"] != int(selected_microbatch):
            raise ValueError("Microbatch certificate selection changed")
        return result
    if time.time() >= deadline_unix - 20:
        raise TimeoutError("No time left for required gradient certificate")
    device = torch.device(config["device"])
    sampler = ConsumedSampler(bundle["train_ids"], config["seeds"]["data_order"])
    identifiers, occurrences = sampler.take(128)
    models, initialization = initialize_models(config, artifact / "gradient_certificate")
    records = {}
    fixture_identity = None
    try:
        with deadline_guard(deadline_unix):
            for kind in ENGINEERING_KINDS:
                initial = copy.deepcopy(models[kind].state_dict())
                gradients_by_micro = {}
                losses = {}
                for microbatch in sorted(set((16, int(selected_microbatch)))):
                    _engineering_release(device)
                    model = models[kind].to(device).train()
                    model.load_state_dict(initial)
                    model.zero_grad(set_to_none=True)
                    weighted_loss = 0.0
                    augments, mask_ids = [], []
                    for offset in range(0, 128, microbatch):
                        if time.time() >= deadline_unix:
                            raise TimeoutError("Gradient certificate deadline")
                        batch = make_batch(bundle["dataset"], identifiers[offset:offset+microbatch],
                                           occurrences[offset:offset+microbatch], config, training=True)
                        augments.extend(batch["augmentation_ids"])
                        mask_ids.extend(batch["mask_ids"])
                        masks = move_masks(batch["masks"], device)
                        images = batch["images"].to(device=device, dtype=torch.float32)
                        output = model(images, masks["context_indices"], masks["context_valid"],
                                       masks["target_indices"], masks["target_valid"])
                        weight = len(images) / 128.0
                        (output["loss"] * weight).backward()
                        weighted_loss += float(output["loss"].detach()) * weight
                        del output, masks, images, batch
                    signature = _engineering_hash({"ids": identifiers, "occurrences": occurrences,
                                                   "augmentation_ids": augments, "mask_ids": mask_ids})
                    if fixture_identity is None:
                        fixture_identity = signature
                    elif signature != fixture_identity:
                        raise AssertionError("Gradient check fixtures changed across microbatch/model")
                    if any(p.grad is not None for p in model.target.parameters()):
                        raise AssertionError("Teacher gradients present")
                    grads = {name: p.grad.detach().cpu().clone() for name, p in model.named_parameters()
                             if p.requires_grad and p.grad is not None}
                    if not grads or not all(bool(torch.isfinite(g).all()) for g in grads.values()):
                        raise AssertionError("Nonfinite or empty gradient")
                    gradients_by_micro[microbatch] = grads
                    losses[microbatch] = weighted_loss
                    model.cpu().zero_grad(set_to_none=True)
                    del model
                    _engineering_release(device)
                baseline, candidate = gradients_by_micro[16], gradients_by_micro[int(selected_microbatch)]
                if baseline.keys() != candidate.keys():
                    raise AssertionError("Gradient parameter names changed")
                max_abs = 0.0
                squares = reference_squares = 0.0
                for name in baseline:
                    torch.testing.assert_close(candidate[name], baseline[name], rtol=3e-4, atol=5e-6,
                                               msg=lambda message, name=name: name + ": " + message)
                    difference = (candidate[name] - baseline[name]).double()
                    max_abs = max(max_abs, float(difference.abs().max()))
                    squares += float(difference.square().sum())
                    reference_squares += float(baseline[name].double().square().sum())
                if abs(losses[16] - losses[int(selected_microbatch)]) > 3e-6:
                    raise AssertionError("Accumulated weighted losses differ")
                records[kind] = {"status": "PASS", "weighted_losses": losses,
                                 "gradient_max_abs_difference": max_abs,
                                 "gradient_relative_l2_difference": (squares/max(reference_squares, 1e-30))**0.5,
                                 "parameter_gradient_tensors": len(baseline),
                                 "teacher_gradients_absent": True, "same_fixture_sha256": signature}
                del gradients_by_micro, baseline, candidate, initial
    finally:
        models.clear()
        _engineering_release(device)
    result = {"schema": ENGINEERING_SCHEMA, "status": "PASS", "selected_microbatch": int(selected_microbatch),
              "baseline_microbatch": 16, "effective_batch": 128, "dtype": "float32",
              "gradient_tolerance": {"rtol": 3e-4, "atol": 5e-6},
              "loss_absolute_tolerance": 3e-6, "fixtures_sha256": fixture_identity,
              "models": records, "sample_count": 128,
              "scope": "Same actual images/augmentation/masks and per-sample weighted JEPA loss, natural packed padding; gradients before clipping/optimizer. This is a numerical equivalence certificate, not bitwise equivalence or a learning result."}
    atomic_json(outpath, result)
    return result


def run_engineering_screen(root, artifact_dir, deadline_unix, config=None, bundle=None):
    """Eight bounded real-image calibration cases; weights discarded, quality never selected.

    Default scope is <8 minutes, further restricted by the parent's absolute deadline.
    Call again with identical inputs to validate/reuse completed cases and resume a
    partial checkpoint. Each eight-update case checkpoints at 0,4,8 with full state.
    """
    root, artifact = Path(root), Path(artifact_dir)
    artifact.mkdir(parents=True, exist_ok=True)
    directory = artifact / "engineering"
    directory.mkdir(exist_ok=True)
    config = _engineering_configuration(root, config)
    bundle = bundle if bundle is not None else reuse_food_bundle(root, config, verify_images=True)
    identity = {"schema": ENGINEERING_SCHEMA, "config": config,
                "source_hashes": definition_hashes(root), "engineering_export_sha256": _engineering_export_hash(root),
                "dataset_fingerprint": bundle["dataset"].fingerprint,
                "split_sha256": bundle["split_manifest"]["split_sha256"],
                "microbatches": [16,32,64,128], "effective_batch": 128,
                "warmup": 3, "measured": 5, "peak_limit_bytes": 70 * 1024**3,
                "selection": "Minimize sum of ViT and semantic median complete step seconds; tie selects smaller microbatch. All successful measured cases within memory limit only. No quality outcomes consulted."}
    identity_hash = _engineering_hash(identity)
    protocol_path = directory / "protocol.json"
    if protocol_path.exists():
        if json.loads(protocol_path.read_text())["identity_hash"] != identity_hash:
            raise ValueError("Engineering identity changed; use a new run ID")
    else:
        atomic_json(protocol_path, {"identity": identity, "identity_hash": identity_hash})
    result_path = artifact / "engineering_screen.json"
    if result_path.exists():
        result = json.loads(result_path.read_text())
        if result["identity_hash"] != identity_hash:
            raise ValueError("Completed engineering result identity changed")
        return result
    started = time.time()
    local_deadline = min(float(deadline_unix), started + 480.0)
    records = []
    initial_models, initialization = initialize_models(config, directory)
    # Keep CPU templates only; each isolated run moves exactly one model to the GPU.
    baseline_traces = None
    try:
        for microbatch in identity["microbatches"]:
            # Alternate architecture order to lessen simple sequential thermal/order effects.
            order = ENGINEERING_KINDS if microbatch in (16,64) else tuple(reversed(ENGINEERING_KINDS))
            for kind in order:
                key = kind + "_mb" + str(microbatch)
                folder = directory / key
                folder.mkdir(exist_ok=True)
                record_path = folder / "case.json"
                local = copy.deepcopy(config)
                local["training"].update(microbatch=microbatch, accumulation_steps=128//microbatch)
                checkpoint = root / "checkpoints" / artifact.name / "engineering" / key
                if record_path.exists():
                    row = json.loads(record_path.read_text())
                    if row["identity_hash"] != identity_hash:
                        raise ValueError("Engineering case identity changed")
                    if row["status"] == "PASS":
                        validate_training_manifest(checkpoint)
                else:
                    row = {"case_key": key, "model_id": kind, "microbatch": microbatch,
                           "accumulation_steps": 128//microbatch, "effective_batch": 128,
                           "status": "NOT_RUN_DEADLINE", "identity_hash": identity_hash}
                    model = result = None
                    try:
                        if time.time() >= local_deadline - 35:
                            raise TimeoutError("Engineering deadline reserve for certificate")
                        _engineering_release(config["device"])
                        model = copy.deepcopy(initial_models[kind])
                        protocol = {"source_hashes": identity["source_hashes"],
                                    "engineering_identity_hash": identity_hash,
                                    "purpose": "real-image microbatch engineering calibration; discarded weights"}
                        with deadline_guard(local_deadline - 25):
                            result = run_model(kind, local, bundle, folder, checkpoint, protocol,
                                               model=model, stop_after=8, deadline_unix=local_deadline-30)
                        manifest = json.loads((checkpoint / "manifest.json").read_text())
                        manifest["resume_notebook"] = "notebooks/04_Bounded_Semantic_Followup.ipynb"
                        manifest["resume_cell"] = "launch-bounded-followup"
                        manifest["resume_instructions"] = "Run Notebook 04 top-to-bottom with the same run ID, source and configuration; this case resumes from the manifest latest valid checkpoint."
                        atomic_json(checkpoint / "manifest.json", manifest)
                        if result["resume"]["final_step"] != 8:
                            raise TimeoutError("Engineering case incomplete")
                        if any(not t["successful"] for t in result["traces"]):
                            raise AssertionError("A calibration update failed")
                        measured = result["metrics"][3:8]
                        if len(measured) != 5 or result["resume"]["actual_image_exposures"] != 1024:
                            raise AssertionError("Calibration exposure/timing count mismatch")
                        traces = [_trace_projection(t) for t in result["traces"]]
                        trace_hash = _engineering_hash(traces)
                        peak = result["performance"]["peak_cuda_allocated_bytes"]
                        row.update(status="PASS", successful_updates=8, image_exposures=1024,
                                   warmup_updates=3, measured_updates=5,
                                   median_step_seconds=float(statistics.median(r["step_seconds"] for r in measured)),
                                   median_compute_seconds=float(statistics.median(r["compute_seconds"] for r in measured)),
                                   median_data_seconds=float(statistics.median(r["data_seconds"] for r in measured)),
                                   raw_step_seconds=[r["step_seconds"] for r in measured],
                                   peak_cuda_allocated_bytes=peak, within_memory_limit=peak <= identity["peak_limit_bytes"],
                                   trace_sha256=trace_hash, checkpoint_dir=str(checkpoint),
                                   performance=result["performance"])
                    except (RuntimeError, FloatingPointError, TimeoutError, AssertionError) as exc:
                        row.update(status="OOM" if isinstance(exc, torch.cuda.OutOfMemoryError) or "out of memory" in str(exc).lower()
                                   else "NOT_RUN_DEADLINE" if isinstance(exc, TimeoutError) else "ERROR",
                                   error=repr(exc))
                    finally:
                        model = result = None
                        _engineering_release(config["device"])
                    atomic_json(record_path, row)
                if row["status"] == "PASS":
                    traces = json.loads((folder / (kind+"_training_traces.json")).read_text())
                    actual_hash = _engineering_hash([_trace_projection(t) for t in traces])
                    if actual_hash != row["trace_sha256"]:
                        raise AssertionError("Saved trace identity mismatch")
                    if baseline_traces is None:
                        baseline_traces = actual_hash
                    elif baseline_traces != actual_hash:
                        raise AssertionError("Samples, occurrences, augmentations or masks differ across cases")
                records.append(row)
                print(f"Engineering {key}: {row['status']}" +
                      (f", {row['median_step_seconds']:.3f} s/update" if row["status"] == "PASS" else ""), flush=True)
    finally:
        initial_models.clear()
        _engineering_release(config["device"])
    candidates = []
    for microbatch in identity["microbatches"]:
        pair = [r for r in records if r["microbatch"] == microbatch and r["status"] == "PASS" and r["within_memory_limit"]]
        if len(pair) == 2 and {r["model_id"] for r in pair} == set(ENGINEERING_KINDS):
            candidates.append((sum(r["median_step_seconds"] for r in pair), microbatch))
    if not candidates:
        raise RuntimeError("No common validated microbatch completed within the engineering budget")
    selected_cost, selected = min(candidates)
    certificate = certify_microbatch_equivalence(root, directory, config, bundle, selected,
                                                 min(float(deadline_unix), max(local_deadline, time.time()+45)))
    result = {"schema": ENGINEERING_SCHEMA, "status": "PASS" if len(records)==8 and all(r["status"]=="PASS" for r in records) else "PARTIAL",
              "identity": identity, "identity_hash": identity_hash, "rows": records,
              "selected_microbatch": selected, "selected_accumulation_steps": 128//selected,
              "selected_pair_median_step_seconds": selected_cost,
              "selected_model_step_seconds": {r["model_id"]:r["median_step_seconds"] for r in records if r["microbatch"]==selected and r["status"]=="PASS"},
              "equivalence_certificate": certificate, "common_trace_sha256": baseline_traces,
              "wall_seconds": time.time()-started, "quality_evidence": False,
              "weights_discarded": True, "heldout_status": "NOT_RUN",
              "timing_scope": "Actual accumulated JEPA training, real data preprocessing/transfers included; checkpoint I/O excluded from step time. 3 warmup + 5 measured updates. One GPU model at a time; FP32 strict deterministic. Small sequential screen, not a formal speed significance test."}
    atomic_json(result_path, result)
    return result


def run_scale_screen(root, artifact_dir, deadline_unix, config=None, token_counts=(256,1024,4096)):
    """Optional <=180-second synthetic scale screen, independent of quality selection.

    Each case is bounded to 25 seconds between complete calls; SIGALRM additionally
    bounds total scope. Timings exclude data and transfer. OOM/TIMEOUT are explicit.
    """
    from fmm_jepa.food101_benchmark import _food_benchmark_case, _food_environment
    root, artifact = Path(root), Path(artifact_dir)
    directory = artifact / "scale_screen"
    directory.mkdir(parents=True, exist_ok=True)
    config = _engineering_configuration(root, config)
    config["training"]["updates"] = 100
    policy = {"microbatch": 2, "warmup": 3, "repeats": 5,
              "seed": int(config["seeds"]["model_init"]), "case_timeout_seconds": 25}
    identity = {"config": config, "policy": policy, "token_counts": list(token_counts),
                "source_hashes": definition_hashes(root), "environment": _food_environment(config),
                "engineering_export_sha256": _engineering_export_hash(root)}
    identity_hash = _engineering_hash(identity)
    started = time.time()
    local_deadline = min(float(deadline_unix), started+180)
    rows=[]
    for tokens in token_counts:
        for scope in ("encoder_forward", "full_jepa_update"):
            for kind in ENGINEERING_KINDS:
                key=f"{tokens}_{scope}_{kind}"
                path=directory/(key+".json")
                if path.exists():
                    row=json.loads(path.read_text())
                    if row["identity_hash"] != identity_hash:
                        raise ValueError("Scale case identity changed")
                else:
                    row={"case_key":key,"model_id":kind,"tokens":int(tokens),"scope":scope,"status":"NOT_RUN_DEADLINE"}
                    model=None
                    try:
                        if time.time()>=local_deadline-5:
                            raise TimeoutError("Scale screen deadline")
                        _engineering_release(config["device"])
                        with isolated_rng(policy["seed"]):
                            model=build_model(config,kind).cpu()
                        case_policy={**policy,"case_timeout_seconds":min(25,local_deadline-time.time())}
                        with deadline_guard(min(local_deadline,time.time()+30)):
                            row=_food_benchmark_case(kind,scope,int(tokens),config,{kind:model},case_policy)
                        row["case_key"]=key
                    except (RuntimeError,FloatingPointError,TimeoutError,AssertionError) as exc:
                        row.update(status="OOM" if isinstance(exc,torch.cuda.OutOfMemoryError) or "out of memory" in str(exc).lower()
                                   else "TIMEOUT" if isinstance(exc,TimeoutError) else "ERROR",error=repr(exc))
                    finally:
                        model=None
                        _engineering_release(config["device"])
                    row["identity_hash"]=identity_hash
                    atomic_json(path,row)
                rows.append(row)
                print(f"Scale {key}: {row['status']}",flush=True)
    for tokens in token_counts:
        for scope in ("encoder_forward","full_jepa_update"):
            pair=[r for r in rows if r["tokens"]==tokens and r["scope"]==scope and r["status"]=="PASS"]
            if len(pair)==2:
                for field in ("image_tensor_sha256","mask_ids","augmentation_ids"):
                    if pair[0][field]!=pair[1][field]:
                        raise AssertionError("Scale fixture mismatch: "+field)
    result={"schema":"semantic-followup-scale/v1","status":"PASS" if all(r["status"]=="PASS" for r in rows) else "PARTIAL",
            "identity":identity,"identity_hash":identity_hash,"rows":rows,"wall_seconds":time.time()-started,
            "quality_evidence":False,"scope":"Synthetic RGB compute only. 3 warmup + 5 timed calls, batch2, one microbatch/update. No natural image quality or unmeasured asymptotic claim."}
    atomic_json(artifact/"scale_screen.json",result)
    return result


In [ ]:
print("Engineering exports ready; GPU work starts only through the Notebook 04 controller.")